# **DEJAVU + PREFETCH + H2O**

Cells copied unchanged (outputs kept) from `archive/OS_MAIN.ipynb`: original cell indices [0, 1, 19, 22, 23, 24, 31, 47, 48, 49, 50].

In [1]:
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
import transformers

transformers.utils.logging.disable_progress_bar()

torch.cuda.set_per_process_memory_fraction(0.45)

# model_name = "gpt2"
model_name = "facebook/opt-6.7b"

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Model: facebook/opt-6.7b
Device: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from datasets import load_dataset

# WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Get long articles by filtering for substantial text
long_prompts = []
for item in dataset:
    text = item["text"].strip()
    tokens = tokenizer.encode(text)
    if len(tokens) > 200:  # only keep long articles
        long_prompts.append(text)
    if len(long_prompts) >= 32:
        break

print(f"Found {len(long_prompts)}  prompts")
for i, p in enumerate(long_prompts[:4]):
    tokens = tokenizer.encode(p)
    print(f"Prompt {i}: {len(tokens)} tokens | {p[:80]}...")

Found 32  prompts
Prompt 0: 217 tokens | He had a recurring role in 2003 on two episodes of The Bill , as character " Con...
Prompt 1: 205 tokens | In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill . The ...
Prompt 2: 246 tokens | Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont...
Prompt 3: 268 tokens | Du Fu 's work is notable above all for its range . Chinese critics traditionally...


In [3]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu_pinned(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk_pinned(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk_pinned(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

In [4]:
def run_h2o_dejavu_prefetch_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    microbatches_count = [8,16,32]
    n_tokens = [256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    num_layers = model.config.num_hidden_layers

    for D in microbatches_count:
        for N in n_tokens:

            prompts = long_prompts[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            # Compute budget per microbatch
            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():
                next_tokens = []
                cpu_kv_caches = []
                cum_scores_list = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    # Initialize cumulative scores
                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)

                    # Offload to pinned CPU memory
                    cpu_kv_caches.append(offload_kv_to_cpu_pinned(outputs.past_key_values))

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch
                with torch.cuda.stream(prefetch_stream):
                    prefetched = tuple(
                        (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                        for k, v in cpu_kv_caches[0]
                    )

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        # Prefetch next microbatch
                        with torch.cuda.stream(prefetch_stream):
                            prefetched = tuple(
                                (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                for k, v in cpu_kv_caches[next_i]
                            )

                        # Build cache and decode
                        gpu_cache = build_kv_cache(gpu_layers)
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True,
                            output_attentions=True
                        )
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        kv = outputs.past_key_values
                        current_len = kv.get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction — one at a time
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv)
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        # Offload evicted cache to pinned CPU
                        with torch.cuda.stream(offload_stream):
                            cpu_kv_caches[i] = tuple(
                                (k.detach().contiguous().cpu().pin_memory(),
                                 v.detach().contiguous().cpu().pin_memory())
                                for k, v in unpack_kv_cache(kv)
                            )

                        del gpu_layers, gpu_cache, outputs, kv

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            cpu_kv_mb = sum(
                k.nelement() * k.element_size() + v.nelement() * v.element_size()
                for cache in cpu_kv_caches for k, v in cache
            ) / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV (MB)": cpu_kv_mb,
            })

            del cpu_kv_caches, next_tokens, cum_scores_list, prefetched
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [5]:
run_h2o_dejavu_prefetch_benchmark()

D=8    N=256  | Budget=217  | Latency=159.574s | Throughput=12.8 tok/s | Peak VRAM=13782.4MB | VRAM%=34.1% | CPU KV=868.0MB
D=8    N=512  | Budget=291  | Latency=303.474s | Throughput=13.5 tok/s | Peak VRAM=13791.1MB | VRAM%=34.1% | CPU KV=1164.0MB
D=16   N=256  | Budget=217  | Latency=280.293s | Throughput=14.6 tok/s | Peak VRAM=13951.7MB | VRAM%=34.5% | CPU KV=1736.0MB
D=16   N=512  | Budget=291  | Latency=606.618s | Throughput=13.5 tok/s | Peak VRAM=13895.8MB | VRAM%=34.4% | CPU KV=2328.0MB
D=32   N=256  | Budget=217  | Latency=569.757s | Throughput=14.4 tok/s | Peak VRAM=14217.0MB | VRAM%=35.2% | CPU KV=3472.0MB
D=32   N=512  | Budget=291  | Latency=1228.247s | Throughput=13.3 tok/s | Peak VRAM=14136.4MB | VRAM%=35.0% | CPU KV=4656.0MB


,Microbatches (D),Tokens Generated (N),Budget,Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV (MB)
0,8,256,217,159.574499,12.834131,13782.368652,34.079872,868.0
1,8,512,291,303.474258,13.497026,13791.126953,34.101528,1164.0
2,16,256,217,280.293100,14.613274,13951.734863,34.498666,1736.0
3,16,512,291,606.617563,13.504390,13895.802246,34.360361,2328.0
4,32,256,217,569.757492,14.378047,14216.997070,35.154584,3472.0
5,32,512,291,1228.247467,13.339331,14136.413086,34.955323,4656.0


In [ ]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

def compute_importance_scores(attentions):
    num_layers = len(attentions)
    num_heads = attentions[0].shape[1]
    num_queries = attentions[0].shape[2]
    num_keys = attentions[0].shape[3]

    scores = torch.zeros(num_keys).to(device)

    for n in range(num_layers):
        layer_attn = attentions[n][0]
        avg_across_heads = layer_attn.sum(dim=0) / num_heads
        avg_across_queries = avg_across_heads.sum(dim=0) / num_queries

        scores += avg_across_queries

    scores = scores / num_layers

    return scores


def classify_indices(top,middle,bottom,scores):
  # sorted scores must be passed here
  sorted_indices = scores.argsort(descending=True)
  top_length = int((len(sorted_indices)/100) * top)
  middle_length = int(((len(sorted_indices)/100)) * middle)
  bottom_lenght = int(((len(sorted_indices)/100)) * bottom)

  hot = sorted_indices[ :top_length]
  cold = sorted_indices[top_length:top_length+middle_length]
  evict = sorted_indices[top_length+middle_length: ]

  return hot,cold,evict

def split_cache(kv_cache, hot_indices, cold_indices):
    hot_layers = []
    cold_layers = []

    for k, v in kv_cache:
        h_idx = hot_indices.to(k.device)
        c_idx = cold_indices.to(k.device)

        hot_k = k[:, :, h_idx, :]
        hot_v = v[:, :, h_idx, :]
        hot_layers.append((hot_k, hot_v))

        cold_k = k[:, :, c_idx, :]
        cold_v = v[:, :, c_idx, :]
        cold_layers.append((cold_k, cold_v))

    return hot_layers, cold_layers

In [ ]:
def run_h2o_dejavu_prefetch_pinned_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    microbatches_count = [4, 8, 16]
    n_tokens = [128, 256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    num_layers = model.config.num_hidden_layers

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            # Compute budget per microbatch
            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():
                next_tokens = []
                cpu_kv_caches = []
                cum_scores_list = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    # Initialize cumulative scores
                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)

                    # Offload to pinned CPU memory
                    cpu_kv_caches.append(offload_kv_to_cpu(outputs.past_key_values))

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch
                with torch.cuda.stream(prefetch_stream):
                    prefetched = tuple(
                        (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                        for k, v in cpu_kv_caches[0]
                    )

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        # Prefetch next microbatch
                        with torch.cuda.stream(prefetch_stream):
                            prefetched = tuple(
                                (k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                for k, v in cpu_kv_caches[next_i]
                            )

                        # Build cache and decode
                        gpu_cache = build_kv_cache(gpu_layers)
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True,
                            output_attentions=True
                        )
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        kv = outputs.past_key_values
                        current_len = kv.get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction — one at a time
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv)
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        # Offload evicted cache to pinned CPU
                        with torch.cuda.stream(offload_stream):
                            cpu_kv_caches[i] = tuple(
                                (k.detach().contiguous().cpu().pin_memory(),
                                 v.detach().contiguous().cpu().pin_memory())
                                for k, v in unpack_kv_cache(kv)
                            )

                        del gpu_layers, gpu_cache, outputs, kv

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            cpu_kv_mb = sum(
                k.nelement() * k.element_size() + v.nelement() * v.element_size()
                for cache in cpu_kv_caches for k, v in cache
            ) / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV (MB)": cpu_kv_mb,
            })

            del cpu_kv_caches, next_tokens, cum_scores_list, prefetched
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [ ]:
run_h2o_dejavu_prefetch_pinned_benchmark()

D=4    N=128  | Budget=74   | Latency=25.926s | Throughput=19.7 tok/s | Peak VRAM=12896.0MB | VRAM%=31.9% | CPU KV=148.0MB
D=4    N=256  | Budget=125  | Latency=56.783s | Throughput=18.0 tok/s | Peak VRAM=13023.9MB | VRAM%=32.2% | CPU KV=250.0MB
D=4    N=512  | Budget=228  | Latency=130.649s | Throughput=15.7 tok/s | Peak VRAM=13294.1MB | VRAM%=32.9% | CPU KV=456.0MB
D=8    N=128  | Budget=74   | Latency=51.110s | Throughput=20.0 tok/s | Peak VRAM=13080.1MB | VRAM%=32.3% | CPU KV=296.0MB
D=8    N=256  | Budget=125  | Latency=113.365s | Throughput=18.1 tok/s | Peak VRAM=13024.0MB | VRAM%=32.2% | CPU KV=500.0MB
D=8    N=512  | Budget=228  | Latency=264.235s | Throughput=15.5 tok/s | Peak VRAM=13294.4MB | VRAM%=32.9% | CPU KV=912.0MB
D=16   N=128  | Budget=74   | Latency=108.898s | Throughput=18.8 tok/s | Peak VRAM=13074.6MB | VRAM%=32.3% | CPU KV=592.0MB
D=16   N=256  | Budget=125  | Latency=236.131s | Throughput=17.3 tok/s | Peak VRAM=13024.0MB | VRAM%=32.2% | CPU KV=1000.0MB
D=16   N=5

,Microbatches (D),Tokens Generated (N),Budget,Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV (MB)
0,4,128,74,25.925889,19.748599,12895.957520,31.888029,148.0
1,4,256,125,56.782836,18.033618,13023.949707,32.204518,250.0
2,4,512,228,130.649469,15.675533,13294.141113,32.872624,456.0
3,8,128,74,51.110460,20.035038,13080.143066,32.343468,296.0
4,8,256,125,113.364562,18.065610,13023.952637,32.204525,500.0
5,8,512,228,264.235304,15.501335,13294.388184,32.873235,912.0
6,16,128,74,108.897896,18.806608,13074.562500,32.329669,592.0
7,16,256,125,236.131009,17.346303,13023.960449,32.204544,1000.0
8,16,512,228,529.534955,15.470178,13294.509277,32.873534,1824.0


In [ ]:
def run_h2o_gpu_only_benchmark(budget_ratio=0.4, recent_ratio=0.5):
    """H2O eviction on GPU only — no CPU offloading, no prefetching."""
    microbatches_count = [4, 8, 16]
    n_tokens = [128, 256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            sample_len = tokenizer(prompts[0], return_tensors="pt").input_ids.shape[1]
            budget = int((sample_len + N) * budget_ratio)
            budget = max(budget, sample_len)
            recent_window = int(budget * recent_ratio)

            total_start = time.perf_counter()

            with torch.no_grad():
                kv_caches = []
                next_tokens = []
                cum_scores_list = []

                # PREFILL
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(
                        input_ids=inputs.input_ids,
                        use_cache=True,
                        output_attentions=True
                    )
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    next_tokens.append(next_token)

                    seq_len = outputs.past_key_values.get_seq_length()
                    scores = torch.zeros(seq_len, device=device)
                    for attn in outputs.attentions:
                        scores += attn[0].sum(dim=0).sum(dim=0)
                    cum_scores_list.append(scores)
                    kv_caches.append(outputs.past_key_values)

                # DECODE
                for step in range(N - 1):
                    for i in range(D):

                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=kv_caches[i],
                            use_cache=True,
                            output_attentions=True
                        )
                        kv_caches[i] = outputs.past_key_values
                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                        # Update cumulative scores
                        current_len = kv_caches[i].get_seq_length()
                        new_score = torch.zeros(1, device=device)
                        cum_scores_list[i] = torch.cat([cum_scores_list[i], new_score])
                        for attn in outputs.attentions:
                            step_scores = attn[0].sum(dim=0).sum(dim=0)
                            if len(step_scores) == len(cum_scores_list[i]):
                                cum_scores_list[i] += step_scores

                        # H2O eviction
                        while current_len > budget:
                            evict_candidates = cum_scores_list[i][:-recent_window].clone()
                            if len(evict_candidates) == 0:
                                break

                            evict_pos = evict_candidates.argmin().item()

                            keep_mask = torch.ones(current_len, dtype=torch.bool, device=device)
                            keep_mask[evict_pos] = False
                            keep_indices = torch.where(keep_mask)[0]

                            kv_layers = unpack_kv_cache(kv_caches[i])
                            kept_layers = [(k[:, :, keep_indices, :], v[:, :, keep_indices, :]) for k, v in kv_layers]
                            kv_caches[i] = build_kv_cache(kept_layers)

                            cum_scores_list[i] = cum_scores_list[i][keep_mask]
                            current_len -= 1

                        del outputs

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100

            kv_bytes = 0
            for cache in kv_caches:
                kv_layers = unpack_kv_cache(cache)
                for k, v in kv_layers:
                    kv_bytes += k.nelement() * k.element_size()
                    kv_bytes += v.nelement() * v.element_size()
            kv_mb = kv_bytes / (1024 ** 2)

            print(f"D={D:<4} N={N:<4} | Budget={budget:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | GPU KV={kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Budget": budget,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "GPU KV (MB)": kv_mb,
            })

            del kv_caches, next_tokens, cum_scores_list
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [ ]:
print("=" * 80)
print("H2O GPU-ONLY (40% budget)")
print("=" * 80)
h2o_gpu_results = run_h2o_gpu_only_benchmark(budget_ratio=0.4)

print("\n" + "=" * 80)
print("H2O + DEJAVU + PREFETCH + PINNED (40% budget)")
print("=" * 80)
h2o_dejavu_results = run_h2o_dejavu_prefetch_pinned_benchmark(budget_ratio=0.4)

H2O GPU-ONLY (40% budget)
D=4    N=128  | Budget=74   | Latency=16.975s | Throughput=30.2 tok/s | Peak VRAM=12933.0MB | VRAM%=32.0% | GPU KV=148.0MB
D=4    N=256  | Budget=125  | Latency=33.405s | Throughput=30.7 tok/s | Peak VRAM=13123.4MB | VRAM%=32.5% | GPU KV=250.0MB
D=4    N=512  | Budget=228  | Latency=66.430s | Throughput=30.8 tok/s | Peak VRAM=13473.4MB | VRAM%=33.3% | GPU KV=456.0MB
D=8    N=128  | Budget=74   | Latency=33.804s | Throughput=30.3 tok/s | Peak VRAM=13249.6MB | VRAM%=32.8% | GPU KV=296.0MB
D=8    N=256  | Budget=125  | Latency=66.486s | Throughput=30.8 tok/s | Peak VRAM=13373.5MB | VRAM%=33.1% | GPU KV=500.0MB
D=8    N=512  | Budget=228  | Latency=131.666s | Throughput=31.1 tok/s | Peak VRAM=13939.8MB | VRAM%=34.5% | GPU KV=912.0MB
D=16   N=128  | Budget=74   | Latency=67.637s | Throughput=30.3 tok/s | Peak VRAM=13513.2MB | VRAM%=33.4% | GPU KV=592.0MB
D=16   N=256  | Budget=125  | Latency=132.916s | Throughput=30.8 tok/s | Peak VRAM=13873.5MB | VRAM%=34.3% | GPU

KeyboardInterrupt: 